# Шаг 8. Прогноз

Прогноз выручки на 30 дней вперёд и честная проверка его точности. Функции из `app/forecast.py`.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

from app.config import SYNTHETIC_DIR
from app.forecast import naive_forecast, to_daily
from app.pipeline import prepare_events


def read_mapping(file_name: str) -> dict[str, str]:
    # Эталон разметки колонок из .json рядом с тестовым файлом
    return json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))


shop, _ = prepare_events(SYNTHETIC_DIR / "shop.xlsx", read_mapping("shop.xlsx"))
salon, _ = prepare_events(SYNTHETIC_DIR / "salon.xlsx", read_mapping("salon.xlsx"))

## 1. to_daily: ряд по дням для прогноза

In [ ]:
# Две колонки: ds - дата, y - выручка за день. Так их требует Prophet
shop_daily = to_daily(shop)
print("Дней:", len(shop_daily))
shop_daily.head()

In [ ]:
# У салона визиты со временем (09:00, 18:00...).
# В ряду по дням время уходит: все визиты одного дня складываются
salon_daily = to_daily(salon)
print("Визиты в салоне:", len(salon), "| дней в ряду:", len(salon_daily))
salon_daily.tail(7)

## 2. naive_forecast: простой прогноз "как на прошлой неделе"

In [ ]:
# Последние 7 дней повторяются по кругу 30 дней вперёд.
# Сравни: первые 7 строк прогноза = последние 7 дней ряда выше
salon_naive = naive_forecast(salon_daily)
salon_naive.head(10)

In [ ]:
# Слабость простого прогноза на графике: случайный всплеск и провал последней недели
# (1 140 в пятницу и 90 во вторник) повторяются КАЖДУЮ неделю. Поэтому он - только точка отсчёта
import matplotlib.pyplot as plt

history = salon_daily.tail(60)
plt.figure(figsize=(10, 4))
plt.plot(history["ds"], history["y"], label="факт")
plt.plot(salon_naive["ds"], salon_naive["yhat"], label="простой прогноз", linestyle="--")
plt.legend()
plt.title("Салон: последние 60 дней и простой прогноз на 30")
plt.grid(alpha=0.3);

### Попробуй сломать

In [ ]:
# Меньше недели истории - повторять нечего
naive_forecast(salon_daily.head(5))